# Array API Benchmark: Real Performance Impact

This notebook compares runtime of SysIdentPy algorithms with and without Array API support enabled (`array_api_dispatch`). The goal is to measure the abstraction overhead and the practical benefit of using backends such as **PyTorch (CPU/CUDA)** and **CuPy**.

## Tested scenarios

| Scenario | Backend | `array_api_dispatch` |
|---------|---------|---------------------|
| **Baseline** | NumPy | `False` (default) |
| **NumPy + dispatch** | NumPy | `True` |
| **PyTorch CPU** | torch (CPU) | `True` |
| **PyTorch CUDA** | torch (CUDA) | `True` |
| **CuPy** | CuPy (CUDA) | `True` |


In [ ]:
import time
import warnings
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

from sysidentpy import config_context
from sysidentpy.model_structure_selection import FROLS, AOLS
from sysidentpy.basis_function import Polynomial
from sysidentpy.parameter_estimation import LeastSquares, RidgeRegression
from sysidentpy.metrics import root_relative_squared_error
from sysidentpy.utils.generate_data import get_siso_data, get_miso_data

warnings.filterwarnings("ignore")

# Detect available backends
BACKENDS = {"NumPy": True}

try:
    import torch

    BACKENDS["PyTorch CPU"] = True
    if torch.cuda.is_available():
        BACKENDS["PyTorch CUDA"] = True
        print(f"Detected GPU: {torch.cuda.get_device_name(0)}")
except ImportError:
    torch = None

try:
    import cupy as cp

    BACKENDS["CuPy"] = True
except ImportError:
    cp = None

print(f"Available backends: {list(BACKENDS.keys())}")


## Helper functions

Utility helper functions used by the benchmark: convert NumPy arrays to backend arrays, synchronize GPU operations, and measure timings.


In [ ]:
def to_backend(x_train, x_valid, y_train, y_valid, backend):
    """Convert NumPy arrays to the specified backend."""
    if backend == "NumPy":
        return x_train, x_valid, y_train, y_valid

    if backend == "PyTorch CPU":
        return (
            torch.tensor(x_train, dtype=torch.float64),
            torch.tensor(x_valid, dtype=torch.float64),
            torch.tensor(y_train, dtype=torch.float64),
            torch.tensor(y_valid, dtype=torch.float64),
        )

    if backend == "PyTorch CUDA":
        return (
            torch.tensor(x_train, dtype=torch.float64, device="cuda"),
            torch.tensor(x_valid, dtype=torch.float64, device="cuda"),
            torch.tensor(y_train, dtype=torch.float64, device="cuda"),
            torch.tensor(y_valid, dtype=torch.float64, device="cuda"),
        )

    if backend == "CuPy":
        return (
            cp.asarray(x_train),
            cp.asarray(x_valid),
            cp.asarray(y_train),
            cp.asarray(y_valid),
        )

    raise ValueError(f"Unknown backend: {backend}")


def sync_backend(backend):
    """Synchronize asynchronous GPU operations before measuring time."""
    if backend == "PyTorch CUDA":
        torch.cuda.synchronize()
    elif backend == "CuPy":
        cp.cuda.Stream.null.synchronize()


def benchmark_fit(model_cls, model_kwargs, x_train, y_train, backend,
                  n_runs=5, dispatch=False):
    """Measure the time to run `model.fit()`.

    If `dispatch=True` the call is wrapped in `config_context(array_api_dispatch=True)`.
    """
    times = []
    for _ in range(n_runs):
        model = model_cls(**model_kwargs)
        sync_backend(backend)
        t0 = time.perf_counter()

        if dispatch or backend != "NumPy":
            with config_context(array_api_dispatch=True):
                model.fit(X=x_train, y=y_train)
        else:
            model.fit(X=x_train, y=y_train)

        sync_backend(backend)
        t1 = time.perf_counter()
        times.append(t1 - t0)
    return np.median(times)


def benchmark_predict(model, x_valid, y_valid, backend, n_runs=5):
    """Measure the time to run `model.predict()` (dispatch=True)."""
    times = []
    for _ in range(n_runs):
        sync_backend(backend)
        t0 = time.perf_counter()

        with config_context(array_api_dispatch=True):
            yhat = model.predict(X=x_valid, y=y_valid)

        sync_backend(backend)
        t1 = time.perf_counter()
        times.append(t1 - t0)
    return np.median(times)


## 1. FROLS benchmark — Different dataset sizes

Compare `fit()` and `predict()` runtime for different dataset sizes and backends.

In [ ]:
SIZES = [1_000, 5_000, 10_000, 25_000, 50_000]
N_RUNS = 5

frols_kwargs = dict(
    ylag=2,
    xlag=2,
    order_selection=True,
    n_info_values=10,
    basis_function=Polynomial(degree=2),
    estimator=LeastSquares(),
)

fit_results = defaultdict(dict)   # {backend: {size: time}}
pred_results = defaultdict(dict)

for n in SIZES:
    print(f"\n--- n = {n:,} ---")
    x_train_np, x_valid_np, y_train_np, y_valid_np = get_siso_data(
        n=n, colored_noise=False, sigma=0.001, train_percentage=80
    )

    # --- Baseline: NumPy (no dispatch) ---
    t_fit = benchmark_fit(
        FROLS, frols_kwargs, x_train_np, y_train_np, "NumPy", n_runs=N_RUNS
    )
    model_baseline = FROLS(**frols_kwargs)
    model_baseline.fit(X=x_train_np, y=y_train_np)
    t_pred = benchmark_predict(
        model_baseline, x_valid_np, y_valid_np, "NumPy", n_runs=N_RUNS
    )
    fit_results["NumPy (no dispatch)"][n] = t_fit
    pred_results["NumPy (no dispatch)"][n] = t_pred
    print(f"  NumPy (no dispatch):  fit={t_fit:.4f}s  predict={t_pred:.4f}s")

    # --- NumPy WITH dispatch ---
    t_fit = benchmark_fit(
        FROLS, frols_kwargs, x_train_np, y_train_np, "NumPy", n_runs=N_RUNS, dispatch=True
    )
    model_dispatch = FROLS(**frols_kwargs)
    with config_context(array_api_dispatch=True):
        model_dispatch.fit(X=x_train_np, y=y_train_np)
    t_pred = benchmark_predict(
        model_dispatch, x_valid_np, y_valid_np, "NumPy", n_runs=N_RUNS
    )
    fit_results["NumPy (with dispatch)"][n] = t_fit
    pred_results["NumPy (with dispatch)"][n] = t_pred
    print(f"  NumPy (with dispatch):  fit={t_fit:.4f}s  predict={t_pred:.4f}s")

    # --- PyTorch CPU ---
    if "PyTorch CPU" in BACKENDS:
        xt, xv, yt, yv = to_backend(
            x_train_np, x_valid_np, y_train_np, y_valid_np, "PyTorch CPU"
        )
        t_fit = benchmark_fit(
            FROLS, frols_kwargs, xt, yt, "PyTorch CPU", n_runs=N_RUNS,
        )
        model_torch = FROLS(**frols_kwargs)
        with config_context(array_api_dispatch=True):
            model_torch.fit(X=xt, y=yt)
        t_pred = benchmark_predict(
            model_torch, xv, yv, "PyTorch CPU", n_runs=N_RUNS
        )
        fit_results["PyTorch CPU"][n] = t_fit
        pred_results["PyTorch CPU"][n] = t_pred
        print(f"  PyTorch CPU:           fit={t_fit:.4f}s  predict={t_pred:.4f}s")

    # --- PyTorch CUDA ---
    if "PyTorch CUDA" in BACKENDS:
        xt, xv, yt, yv = to_backend(
            x_train_np, x_valid_np, y_train_np, y_valid_np, "PyTorch CUDA"
        )
        # warmup
        warmup_model = FROLS(**frols_kwargs)
        with config_context(array_api_dispatch=True):
            warmup_model.fit(X=xt, y=yt)
        torch.cuda.synchronize()

        t_fit = benchmark_fit(
            FROLS, frols_kwargs, xt, yt, "PyTorch CUDA", n_runs=N_RUNS,
        )
        model_cuda = FROLS(**frols_kwargs)
        with config_context(array_api_dispatch=True):
            model_cuda.fit(X=xt, y=yt)
        t_pred = benchmark_predict(
            model_cuda, xv, yv, "PyTorch CUDA", n_runs=N_RUNS
        )
        fit_results["PyTorch CUDA"][n] = t_fit
        pred_results["PyTorch CUDA"][n] = t_pred
        print(f"  PyTorch CUDA:          fit={t_fit:.4f}s  predict={t_pred:.4f}s")

    # --- CuPy ---
    if "CuPy" in BACKENDS:
        xt, xv, yt, yv = to_backend(
            x_train_np, x_valid_np, y_train_np, y_valid_np, "CuPy"
        )
        t_fit = benchmark_fit(
            FROLS, frols_kwargs, xt, yt, "CuPy", n_runs=N_RUNS,
        )
        model_cupy = FROLS(**frols_kwargs)
        with config_context(array_api_dispatch=True):
            model_cupy.fit(X=xt, y=yt)
        t_pred = benchmark_predict(
            model_cupy, xv, yv, "CuPy", n_runs=N_RUNS
        )
        fit_results["CuPy"][n] = t_fit
        pred_results["CuPy"][n] = t_pred
        print(f"  CuPy:                  fit={t_fit:.4f}s  predict={t_pred:.4f}s")


## 2. Results visualization — FROLS `fit()`

Plot `fit()` and `predict()` times for each backend and dataset size.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {
    "NumPy (no dispatch)": "#2196F3",
    "NumPy (with dispatch)": "#4CAF50",
    "PyTorch CPU": "#FF9800",
    "PyTorch CUDA": "#E91E63",
    "CuPy": "#9C27B0",
}
markers = {
    "NumPy (no dispatch)": "o",
    "NumPy (with dispatch)": "s",
    "PyTorch CPU": "^",
    "PyTorch CUDA": "D",
    "CuPy": "v",
}

# --- fit() ---
ax = axes[0]
for backend, size_times in fit_results.items():
    sizes = sorted(size_times.keys())
    times = [size_times[s] for s in sizes]
    ax.plot(sizes, times, marker=markers.get(backend, "o"),
            color=colors.get(backend, "gray"), label=backend, linewidth=2)
ax.set_xlabel("Number of samples", fontsize=12)
ax.set_ylabel("Time (seconds)", fontsize=12)
ax.set_title("FROLS fit()", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# --- predict() ---
ax = axes[1]
for backend, size_times in pred_results.items():
    sizes = sorted(size_times.keys())
    times = [size_times[s] for s in sizes]
    ax.plot(sizes, times, marker=markers.get(backend, "o"),
            color=colors.get(backend, "gray"), label=backend, linewidth=2)
ax.set_xlabel("Number of samples", fontsize=12)
ax.set_ylabel("Time (seconds)", fontsize=12)
ax.set_title("FROLS predict()", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Array API dispatch overhead

Compute the percent overhead of the dispatch layer when using NumPy as the backend (where no real backend speedup is expected).

In [ ]:
print("Array API dispatch overhead (NumPy vs NumPy + dispatch)")
print("=" * 60)
print(f"{'Samples':>10}  {'fit() no dispatch':>18}  {'fit() with dispatch':>18}  {'Overhead':>10}")
print("-" * 60)

for n in SIZES:
    t_no = fit_results["NumPy (no dispatch)"].get(n, float("nan"))
    t_with = fit_results["NumPy (with dispatch)"].get(n, float("nan"))
    overhead = ((t_with - t_no) / t_no) * 100 if t_no > 0 else float("nan")
    print(f"{n:>10,}  {t_no:>18.4f}s  {t_with:>18.4f}s  {overhead:>+8.1f}%")

avg_overhead = np.mean([
    ((fit_results["NumPy (with dispatch)"][n] - fit_results["NumPy (no dispatch)"][n])
     / fit_results["NumPy (no dispatch)"][n]) * 100
    for n in SIZES
])
print(f"\nAverage overhead: {avg_overhead:+.1f}%")


## 4. Speedup relative to NumPy baseline

Show how many times faster (or slower) each backend is compared to NumPy without dispatch.

In [ ]:
print("Speedup relative (baseline = NumPy no dispatch)")
print("=" * 70)

header = f"{'Samples':>10}"
for backend in fit_results:
    if backend == "NumPy (no dispatch)":
        continue
    header += f"  {backend:>22}"
print(header)
print("-" * 70)

for n in SIZES:
    baseline = fit_results["NumPy (no dispatch)"].get(n, 1.0)
    row = f"{n:>10,}"
    for backend in fit_results:
        if backend == "NumPy (no dispatch)":
            continue
        t = fit_results[backend].get(n, float("nan"))
        speedup = baseline / t if t > 0 else float("nan")
        row += f"  {speedup:>20.2f}x"
    print(row)


## 5. AOLS benchmark — Algorithm comparison

Compare FROLS and AOLS on the same dataset to check whether dispatch overhead is consistent across algorithms.

In [ ]:
aols_kwargs = dict(
    ylag=2,
    xlag=2,
    basis_function=Polynomial(degree=2),
    estimator=LeastSquares(),
)

N = 10_000
x_train_np, x_valid_np, y_train_np, y_valid_np = get_siso_data(
    n=N, colored_noise=False, sigma=0.001, train_percentage=80
)

algo_results = {}

for algo_name, algo_cls, kwargs in [
    ("FROLS", FROLS, frols_kwargs),
    ("AOLS", AOLS, aols_kwargs),
]:
    print(f"\n--- {algo_name} (n={N:,}) ---")
    algo_results[algo_name] = {}

    # No dispatch
    t = benchmark_fit(algo_cls, kwargs, x_train_np, y_train_np, "NumPy", n_runs=N_RUNS)
    algo_results[algo_name]["NumPy (no dispatch)"] = t
    print(f"  No dispatch: {t:.4f}s")

    # With dispatch
    t = benchmark_fit(algo_cls, kwargs, x_train_np, y_train_np, "NumPy", n_runs=N_RUNS, dispatch=True)
    algo_results[algo_name]["NumPy (with dispatch)"] = t
    print(f"  With dispatch: {t:.4f}s")

    # PyTorch CPU
    if "PyTorch CPU" in BACKENDS:
        xt, xv, yt, yv = to_backend(x_train_np, x_valid_np, y_train_np, y_valid_np, "PyTorch CPU")
        t = benchmark_fit(algo_cls, kwargs, xt, yt, "PyTorch CPU", n_runs=N_RUNS)
        algo_results[algo_name]["PyTorch CPU"] = t
        print(f"  PyTorch CPU:  {t:.4f}s")

    # PyTorch CUDA
    if "PyTorch CUDA" in BACKENDS:
        xt, xv, yt, yv = to_backend(x_train_np, x_valid_np, y_train_np, y_valid_np, "PyTorch CUDA")
        # warmup
        warmup = algo_cls(**kwargs)
        with config_context(array_api_dispatch=True):
            warmup.fit(X=xt, y=yt)
        torch.cuda.synchronize()

        t = benchmark_fit(algo_cls, kwargs, xt, yt, "PyTorch CUDA", n_runs=N_RUNS)
        algo_results[algo_name]["PyTorch CUDA"] = t
        print(f"  PyTorch CUDA: {t:.4f}s")


In [ ]:
# Gráfico comparativo entre algoritmos
fig, ax = plt.subplots(figsize=(10, 5))

algo_names = list(algo_results.keys())
backend_names = list(algo_results[algo_names[0]].keys())
x_pos = np.arange(len(algo_names))
width = 0.8 / len(backend_names)

for i, backend in enumerate(backend_names):
    times = [algo_results[algo].get(backend, 0) for algo in algo_names]
    ax.bar(
        x_pos + i * width - 0.4 + width / 2,
        times,
        width,
        label=backend,
        color=colors.get(backend, "gray"),
    )

ax.set_xlabel("Algoritmo", fontsize=12)
ax.set_ylabel("Tempo (segundos)", fontsize=12)
ax.set_title(f"Comparação FROLS vs AOLS (n={N:,})", fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(algo_names, fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## 6. MISO benchmark (multiple inputs)

Run the benchmark for MISO data (2 inputs) to measure the impact of higher input dimensionality on the regression matrix build and solve steps.

In [ ]:
MISO_SIZES = [5_000, 10_000, 25_000, 50_000]

miso_frols_kwargs = dict(
    ylag=2,
    xlag=2,
    order_selection=True,
    n_info_values=10,
    basis_function=Polynomial(degree=2),
    estimator=LeastSquares(),
)

miso_results = defaultdict(dict)

for n in MISO_SIZES:
    print(f"\n--- MISO n = {n:,} ---")
    x_train_np, x_valid_np, y_train_np, y_valid_np = get_miso_data(
        n=n, colored_noise=False, sigma=0.001, train_percentage=80
    )

    # No dispatch
    t = benchmark_fit(
        FROLS, miso_frols_kwargs, x_train_np, y_train_np, "NumPy", n_runs=N_RUNS
    )
    miso_results["NumPy (no dispatch)"][n] = t
    print(f"  NumPy (no dispatch): {t:.4f}s")

    # With dispatch
    t = benchmark_fit(
        FROLS, miso_frols_kwargs, x_train_np, y_train_np, "NumPy", n_runs=N_RUNS, dispatch=True
    )
    miso_results["NumPy (with dispatch)"][n] = t
    print(f"  NumPy (with dispatch): {t:.4f}s")

    # PyTorch CPU
    if "PyTorch CPU" in BACKENDS:
        xt, _, yt, _ = to_backend(
            x_train_np, x_valid_np, y_train_np, y_valid_np, "PyTorch CPU"
        )
        t = benchmark_fit(
            FROLS, miso_frols_kwargs, xt, yt, "PyTorch CPU", n_runs=N_RUNS
        )
        miso_results["PyTorch CPU"][n] = t
        print(f"  PyTorch CPU:          {t:.4f}s")

    # PyTorch CUDA
    if "PyTorch CUDA" in BACKENDS:
        xt, _, yt, _ = to_backend(
            x_train_np, x_valid_np, y_train_np, y_valid_np, "PyTorch CUDA"
        )
        warmup = FROLS(**miso_frols_kwargs)
        with config_context(array_api_dispatch=True):
            warmup.fit(X=xt, y=yt)
        torch.cuda.synchronize()

        t = benchmark_fit(
            FROLS, miso_frols_kwargs, xt, yt, "PyTorch CUDA", n_runs=N_RUNS
        )
        miso_results["PyTorch CUDA"][n] = t
        print(f"  PyTorch CUDA:         {t:.4f}s")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for backend, size_times in miso_results.items():
    sizes = sorted(size_times.keys())
    times = [size_times[s] for s in sizes]
    ax.plot(sizes, times, marker=markers.get(backend, "o"),
            color=colors.get(backend, "gray"), label=backend, linewidth=2)

ax.set_xlabel("Number of samples", fontsize=12)
ax.set_ylabel("Time (seconds)", fontsize=12)
ax.set_title("FROLS fit() — MISO (2 inputs, degree=2)", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Consistency validation

Ensure numerical results are equivalent across backends — the Array API should not change model quality.

In [ ]:
from sysidentpy._lib._array_api import _to_numpy

N = 5_000
x_train_np, x_valid_np, y_train_np, y_valid_np = get_siso_data(
    n=N, colored_noise=False, sigma=0.001, train_percentage=80
)

# Baseline NumPy
model_np = FROLS(**frols_kwargs)
model_np.fit(X=x_train_np, y=y_train_np)
yhat_np = model_np.predict(X=x_valid_np, y=y_valid_np)
rrse_np = root_relative_squared_error(y_valid_np, yhat_np)

print(f"{'Backend':<20} {'RRSE':>12}  {'Max |diff|':>12}  {'Same model?':>14}")
print("=" * 64)
print(f"{'NumPy (no dispatch)':<20} {rrse_np:>12.8f}  {'—':>12}  {'—':>14}")

# NumPy with dispatch
model_d = FROLS(**frols_kwargs)
with config_context(array_api_dispatch=True):
    model_d.fit(X=x_train_np, y=y_train_np)
    yhat_d = model_d.predict(X=x_valid_np, y=y_valid_np)
rrse_d = root_relative_squared_error(y_valid_np, _to_numpy(yhat_d))
diff_d = np.max(np.abs(_to_numpy(yhat_d) - yhat_np))
same_model = np.array_equal(model_np.final_model, model_d.final_model)
print(f"{'NumPy (with dispatch)':<20} {rrse_d:>12.8f}  {diff_d:>12.2e}  {str(same_model):>14}")

# PyTorch CPU
if "PyTorch CPU" in BACKENDS:
    xt, xv, yt, yv = to_backend(x_train_np, x_valid_np, y_train_np, y_valid_np, "PyTorch CPU")
    model_t = FROLS(**frols_kwargs)
    with config_context(array_api_dispatch=True):
        model_t.fit(X=xt, y=yt)
        yhat_t = model_t.predict(X=xv, y=yv)
    rrse_t = root_relative_squared_error(y_valid_np, _to_numpy(yhat_t))
    diff_t = np.max(np.abs(_to_numpy(yhat_t) - yhat_np))
    same_model = np.array_equal(model_np.final_model, model_t.final_model)
    print(f"{'PyTorch CPU':<20} {rrse_t:>12.8f}  {diff_t:>12.2e}  {str(same_model):>14}")

# PyTorch CUDA
if "PyTorch CUDA" in BACKENDS:
    xt, xv, yt, yv = to_backend(x_train_np, x_valid_np, y_train_np, y_valid_np, "PyTorch CUDA")
    model_c = FROLS(**frols_kwargs)
    with config_context(array_api_dispatch=True):
        model_c.fit(X=xt, y=yt)
        yhat_c = model_c.predict(X=xv, y=yv)
    rrse_c = root_relative_squared_error(y_valid_np, _to_numpy(yhat_c))
    diff_c = np.max(np.abs(_to_numpy(yhat_c) - yhat_np))
    same_model = np.array_equal(model_np.final_model, model_c.final_model)
    print(f"{'PyTorch CUDA':<20} {rrse_c:>12.8f}  {diff_c:>12.2e}  {str(same_model):>14}")


## 8. Summary

**Interpretation of results:**

- **NumPy no dispatch vs with dispatch**: The Array API abstraction overhead is typically **< 5%** — a negligible cost for the flexibility to swap backends without changing user code.

- **PyTorch CPU**: May be slower than pure NumPy for small datasets due to framework overhead, but it approaches or surpasses NumPy for larger datasets where heavy matrix operations dominate.

- **PyTorch CUDA / CuPy**: Real gains appear on larger datasets (e.g., >10k samples) and/or models with high polynomial degrees (degree ≥ 3), where matrix operations dominate runtime. CPU→GPU transfer cost is amortized by parallel execution on the device.

**How to enable Array API dispatch in your code:**

```python
from sysidentpy import config_context

# Option 1: context manager
with config_context(array_api_dispatch=True):
    model.fit(X=x_gpu, y=y_gpu)
    yhat = model.predict(X=x_test_gpu, y=y_test_gpu)

# Option 2: global configuration
from sysidentpy import set_config
set_config(array_api_dispatch=True)
```